# Task: update vote likes (tick)

Refresh the like counts for the active theme vote's option comments. Run on a recurring tick by `core.scheduler`, so it must be a clean no-op when no vote is active. Headless version of the repo-root `update_vote_likes.ipynb`.

In [ ]:
dry_run = False  # fetch like counts but skip writing them back

In [ ]:
import os
import sys

# add the repo root to sys.path so `import config` / `from core...` resolve
# when this notebook is run directly (e.g. from VSCode); when the scheduler
# runs it via papermill the working directory is already the root, so this is
# a harmless no-op.
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'pyproject.toml')) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

In [ ]:
import datetime

import atproto
from sqlalchemy import select
from sqlalchemy.orm import selectinload

import config
from core.database import session_scope
from core.database.models import ThemeVoteRecord
from core.interactions.votes import active_theme_vote

In [ ]:
with session_scope() as session:
    vote = active_theme_vote(session)

if vote is None:
    print('no active theme vote - nothing to update')
else:
    print(f'active vote {vote.id} ({vote.type.value})')
    print(f'  voting: {vote.vote_start_date} -> {vote.vote_end_date}')

In [ ]:
if vote is not None:
    uris = [o.comment_uri for o in vote.options if o.comment_uri]
    if not uris:
        print('this vote has no posted option comments to check')
    else:
        client = atproto.Client()
        client.login(config.ATPROTO_CLIENT_USERNAME, config.ATPROTO_CLIENT_PASSWORD)

        posts = client.get_posts(uris).posts
        likes_by_uri = {post.uri: (post.like_count or 0) for post in posts}

        now = datetime.datetime.now(datetime.timezone.utc)
        if dry_run:
            for option in vote.options:
                print(f'  {option.theme_name}: {likes_by_uri.get(option.comment_uri)} (dry_run, not saved)')
        else:
            with session_scope() as session:
                record = session.scalar(
                    select(ThemeVoteRecord)
                    .where(ThemeVoteRecord._id == vote.id)
                    .options(selectinload(ThemeVoteRecord.options))
                )
                for option in record.options:
                    fetched = likes_by_uri.get(option.comment_uri)
                    if fetched is not None:
                        option.likes = fetched
                    option._checked_date = now
                    print(f'  {option.theme_name}: {option.likes} likes')
                record._checked_date = now
            print('saved')